# OmniVoice Fine-tune 1 Giọng Nói (Colab)

Notebook này giúp bạn fine-tune 1 giọng riêng bằng OmniVoice.

## Chuẩn bị dữ liệu
- Mỗi sample gồm 1 file wav + 1 file txt cùng tên.
- Ví dụ: `001.wav` và `001.txt`.
- Khuyên dùng ít nhất 20 sample (càng nhiều càng tốt).
- Đặt dữ liệu trong zip theo cấu trúc:

```
dataset/
  audio/
    001.wav
    002.wav
  text/
    001.txt
    002.txt
```


In [ ]:
#@title 1) Check GPU
!nvidia-smi

In [ ]:
#@title 2) Clone repo + install deps
%cd /content
!git clone https://github.com/hieu09030/omnivoice-voice-clone-kit.git
%cd /content/omnivoice-voice-clone-kit
!python -m pip install --upgrade pip
!pip install -r requirements.txt accelerate

In [ ]:
#@title 3) Upload zip dataset
# Zip nên chứa 2 thư mục con: audio/ và text/
from google.colab import files
uploaded = files.upload()

In [ ]:
#@title 4) Giải nén dataset vào data/raw
import os
import zipfile
from pathlib import Path

zip_name = next(iter(uploaded.keys()))
extract_root = Path('/content/omnivoice-voice-clone-kit/data/raw')
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(extract_root)

print('Extracted to:', extract_root)
for p in sorted(extract_root.rglob('*'))[:40]:
    print(p)

In [ ]:
#@title 5) Tự tìm audio_dir và text_dir
from pathlib import Path

root = Path('/content/omnivoice-voice-clone-kit/data/raw')
audio_candidates = [p for p in root.rglob('*') if p.is_dir() and p.name.lower() in ('audio', 'audios', 'wav', 'wavs')]
text_candidates = [p for p in root.rglob('*') if p.is_dir() and p.name.lower() in ('text', 'texts', 'txt', 'transcript', 'transcripts')]

if not audio_candidates or not text_candidates:
    raise RuntimeError('Không tìm thấy thư mục audio/text. Hãy kiểm tra lại cấu trúc zip.')

audio_dir = str(audio_candidates[0])
text_dir = str(text_candidates[0])

print('audio_dir =', audio_dir)
print('text_dir  =', text_dir)

In [ ]:
#@title 6) Tạo train/dev manifest
LANGUAGE_ID = 'vi'  #@param {type:'string'}
DEV_RATIO = 0.05    #@param {type:'number'}

!python finetune/prepare_manifest.py \
  --audio_dir "{audio_dir}" \
  --text_dir "{text_dir}" \
  --out_dir data/finetune/manifests \
  --language_id "{LANGUAGE_ID}" \
  --dev_ratio {DEV_RATIO}

In [ ]:
#@title 7) Chỉnh train config cho Colab GPU (fp16, steps)
import json

STEPS = 1200  #@param {type:'integer'}
LEARNING_RATE = 1e-5  #@param {type:'number'}

cfg_path = 'finetune/train_config_finetune_sdpa.json'
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

cfg['mixed_precision'] = 'fp16'
cfg['steps'] = int(STEPS)
cfg['learning_rate'] = float(LEARNING_RATE)

with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print('Updated config:')
print(json.dumps({k: cfg[k] for k in ['mixed_precision', 'steps', 'learning_rate']}, indent=2))

In [ ]:
#@title 8) Stage 1: Extract audio tokens
!python -m omnivoice.scripts.extract_audio_tokens \
  --input_jsonl data/finetune/manifests/train.jsonl \
  --tar_output_pattern data/finetune/tokens/train/audios/shard-%06d.tar \
  --jsonl_output_pattern data/finetune/tokens/train/txts/shard-%06d.jsonl \
  --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
  --nj_per_gpu 2 \
  --shuffle True

!python -m omnivoice.scripts.extract_audio_tokens \
  --input_jsonl data/finetune/manifests/dev.jsonl \
  --tar_output_pattern data/finetune/tokens/dev/audios/shard-%06d.tar \
  --jsonl_output_pattern data/finetune/tokens/dev/txts/shard-%06d.jsonl \
  --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
  --nj_per_gpu 2 \
  --shuffle False

In [ ]:
#@title 9) Stage 2: Fine-tune
!accelerate launch --num_processes 1 -m omnivoice.cli.train \
  --train_config finetune/train_config_finetune_sdpa.json \
  --data_config finetune/data_config_finetune.json \
  --output_dir exp/voice_clone_finetune

In [ ]:
#@title 10) Test infer bằng checkpoint mới nhất
import glob
import os

TEST_TEXT = 'Xin chào, đây là giọng nói sau khi fine tune.'  #@param {type:'string'}
REF_AUDIO = ''  #@param {type:'string'}

ckpts = sorted(glob.glob('exp/voice_clone_finetune/checkpoint-*'))
if not ckpts:
    raise RuntimeError('Không tìm thấy checkpoint nào trong exp/voice_clone_finetune')
latest_ckpt = ckpts[-1]
print('Using checkpoint:', latest_ckpt)

if not REF_AUDIO:
    raise RuntimeError('Hãy điền REF_AUDIO (đường dẫn wav tham chiếu) trước khi chạy cell này.')

!python clone_tts.py \
  --model "{latest_ckpt}" \
  --text "{TEST_TEXT}" \
  --ref_audio "{REF_AUDIO}" \
  --output out_finetuned.wav \
  --device cuda

print('Output file: /content/omnivoice-voice-clone-kit/out_finetuned.wav')